# Synthetic Pest Video — Batch Renderer
Run this notebook on **Google Colab (CPU runtime)**.
Open 5 copies simultaneously, change `SESSION_ID` to 0–4 in each one.

**Before running:** Upload to Google Drive:
- `pest_project/kitchens/`     ← `generator/kitchen_img/curated_img/` contents
- `pest_project/depth_cache/`  ← `depth_cache/` contents
- `pest_project/code.zip`      ← zipped `synthetic_video_gen/` repo

In [ ]:
# ── CHANGE THIS IN EACH TAB ──────────────────────────────────────────────────
SESSION_ID      = 1  # 0, 1, 2, 3, or 4
TOTAL_SESSIONS  = 5
VIDEOS_PER_KITCHEN = 20
SPLIT_SEED      = 42  # must be same across all sessions
# ─────────────────────────────────────────────────────────────────────────────

DRIVE_ROOT  = '/content/drive/MyDrive/pest_project'
IMAGE_DIR   = f'{DRIVE_ROOT}/curated_img'
DEPTH_CACHE = f'{DRIVE_ROOT}/depth_cache'
OUTPUT_DIR  = f'{DRIVE_ROOT}/renders'
CODE_ZIP    = f'{DRIVE_ROOT}/code.zip'
CODE_DIR    = '/content/synthetic_video_gen'

print(f'Session {SESSION_ID + 1} of {TOTAL_SESSIONS}')


Session 2 of 5


In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [ ]:
import json, random
from pathlib import Path

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
HELD_OUT_FRAC  = 0.10
VAL_FRAC       = 0.10
TOTAL_SESSIONS = 5
SEED           = 42
IMAGE_DIR      = Path("/content/drive/MyDrive/pest_project/curated_img") # Updated to match your Drive
MANIFEST_PATH  = Path("/content/drive/MyDrive/pest_project/renders/manifest.json")

# Exact same logic as render_batch_colab.py
kitchens = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in SUPPORTED_EXTS)
rng = random.Random(SEED)
shuffled = list(kitchens)
rng.shuffle(shuffled)
n = len(shuffled)
n_held  = max(1, round(n * HELD_OUT_FRAC))
n_val   = max(1, round(n * VAL_FRAC))
n_train = n - n_held - n_val

splits = {
    "train":    shuffled[:n_train],
    "val":      shuffled[n_train:n_train + n_val],
    "held_out": shuffled[n_train + n_val:],
}
active_kitchens = splits["train"] + splits["val"]  # same order as render script

# Ensure manifest file exists before attempting to read
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
else:
    manifest = {}
    print(f"Warning: Manifest file not found at {MANIFEST_PATH}. Initializing with empty manifest for this cell's checks.")

done_keys = set(manifest.get("kitchens", {}).keys())

print(f"Total kitchens: {len(kitchens)}")
print(f"  Train: {len(splits['train'])}, Val: {len(splits['val'])}, Held-out: {len(splits['held_out'])}")
print(f"  Renderable (train+val): {len(active_kitchens)}")
print(f"\nIn manifest: {len(done_keys)}")

missing = [k for k in active_kitchens if k.name not in done_keys]
print(f"Missing from manifest: {len(missing)}")
print(f"Videos remaining: {len(missing) * 20}")

session_missing = {i: [] for i in range(TOTAL_SESSIONS)}
for i, k in enumerate(active_kitchens):
    if k.name not in done_keys:
        session_missing[i % TOTAL_SESSIONS].append(k.name)

print(f"\nMissing kitchens per session (correct assignment):")
for sid, ks in session_missing.items():
    print(f"  Session {sid}: {len(ks)} missing  {'← needs re-run' if ks else '✓ done'}")
    for name in ks:
        print(f"    {name}")


Total kitchens: 133
  Train: 107, Val: 13, Held-out: 13
  Renderable (train+val): 120

In manifest: 120
Missing from manifest: 0
Videos remaining: 0

Missing kitchens per session (correct assignment):
  Session 0: 0 missing  ✓ done
  Session 1: 0 missing  ✓ done
  Session 2: 0 missing  ✓ done
  Session 3: 0 missing  ✓ done
  Session 4: 0 missing  ✓ done


# New Section

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────────────────────
# Run once per session. Takes ~2 min.
!pip install -q mmengine timm scipy
!pip install -q 'opencv-python-headless' Pillow
print('Dependencies installed.')

Dependencies installed.


In [ ]:
# ── 3. Unzip repo code ───────────────────────────────────────────────────────
import os, zipfile

if not os.path.exists(CODE_DIR):
    print('Unzipping code...')
    with zipfile.ZipFile(CODE_ZIP, 'r') as z:
        z.extractall('/content')
    # Handle if zip contains a top-level folder
    extracted = [d for d in os.listdir('/content') if 'synthetic_video_gen' in d.lower()]
    if extracted and extracted[0] != 'synthetic_video_gen':
        os.rename(f'/content/{extracted[0]}', CODE_DIR)
    print(f'Code extracted to {CODE_DIR}')
else:
    print('Code already present.')

# Verify key files exist
for f in ['scripts/render_batch_colab.py', 'generator/pipeline.py', 'generator/mmcv_stub']:
    status = '✓' if os.path.exists(f'{CODE_DIR}/{f}') else '✗ MISSING'
    print(f'  {status}  {f}')

Code already present.
  ✓  scripts/render_batch_colab.py
  ✓  generator/pipeline.py
  ✓  generator/mmcv_stub


In [ ]:
# ── 4. Verify Drive folders ──────────────────────────────────────────────────
import os

kitchens = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]
caches   = [f for f in os.listdir(DEPTH_CACHE) if f.endswith('.npz')]

print(f'Kitchen images:  {len(kitchens)}')
print(f'Depth caches:    {len(caches)}')

if len(kitchens) == 0:
    raise RuntimeError(f'No images found in {IMAGE_DIR} — check Drive upload')
if len(caches) < len(kitchens) * 0.9:
    print(f'WARNING: only {len(caches)} caches for {len(kitchens)} images — some renders will run Metric3D (slow)')

Kitchen images:  133
Depth caches:    133


### Verify Google Drive content

Run the following cells to check if the `pest_project`, `kitchens`, and `depth_cache` directories exist in your Google Drive at the expected locations. If they don't appear, you will need to upload them according to the instructions in the first cell.

In [ ]:
import os

print(f"Checking for DRIVE_ROOT: {DRIVE_ROOT}")
if os.path.exists(DRIVE_ROOT):
    print(f"'{DRIVE_ROOT}' exists. Contents:")
    for item in os.listdir(DRIVE_ROOT):
        print(f"  - {item}")
else:
    print(f"ERROR: '{DRIVE_ROOT}' does not exist. Please ensure 'pest_project' is directly in 'My Drive'.")

print(f"\nChecking for IMAGE_DIR: {IMAGE_DIR}")
if os.path.exists(IMAGE_DIR):
    print(f"'{IMAGE_DIR}' exists. Number of files/folders inside: {len(os.listdir(IMAGE_DIR))}")
    # Optionally print a few items if the list is not too long
    if len(os.listdir(IMAGE_DIR)) > 0: print(f"  First 5 items: {os.listdir(IMAGE_DIR)[:5]}")
else:
    print(f"ERROR: '{IMAGE_DIR}' does not exist. Ensure 'kitchens' is inside 'pest_project'.")

print(f"\nChecking for DEPTH_CACHE: {DEPTH_CACHE}")
if os.path.exists(DEPTH_CACHE):
    print(f"'{DEPTH_CACHE}' exists. Number of files/folders inside: {len(os.listdir(DEPTH_CACHE))}")
    if len(os.listdir(DEPTH_CACHE)) > 0: print(f"  First 5 items: {os.listdir(DEPTH_CACHE)[:5]}")
else:
    print(f"ERROR: '{DEPTH_CACHE}' does not exist. Ensure 'depth_cache' is inside 'pest_project'.")


Checking for DRIVE_ROOT: /content/drive/MyDrive/pest_project
'/content/drive/MyDrive/pest_project' exists. Contents:
  - curated_img
  - depth_cache
  - code.zip
  - renders

Checking for IMAGE_DIR: /content/drive/MyDrive/pest_project/curated_img
'/content/drive/MyDrive/pest_project/curated_img' exists. Number of files/folders inside: 135
  First 5 items: ['.gitkeep', '.DS_Store', 'kitchen_0002.jpg', 'kitchen_0003.jpg', 'kitchen_0004.jpg']

Checking for DEPTH_CACHE: /content/drive/MyDrive/pest_project/depth_cache
'/content/drive/MyDrive/pest_project/depth_cache' exists. Number of files/folders inside: 134
  First 5 items: ['kitchen_0002.npz', 'kitchen_0003.npz', 'kitchen_0004.npz', 'kitchen_0005.npz', 'kitchen_0006.npz']


After verifying that the `kitchens` and `depth_cache` folders exist in your Google Drive at `'/content/drive/MyDrive/pest_project/'`, please re-run the cell above this (cell `lAYjDbNNkRRr`) to continue with the notebook.

In [ ]:
# ── 5. Preview kitchen split (optional, no renders) ──────────────────────────
!cd {CODE_DIR} && PYTHONPATH="generator/mmcv_stub:." python scripts/render_batch_colab.py \
    --image_dir   {IMAGE_DIR} \
    --depth_cache {DEPTH_CACHE} \
    --output_dir  {OUTPUT_DIR} \
    --n           {VIDEOS_PER_KITCHEN} \
    --seed        {SPLIT_SEED} \
    --session_id  {SESSION_ID} \
    --total_sessions {TOTAL_SESSIONS} \
    --list_splits


train (107 kitchens):
  kitchen_0033.jpg
  kitchen_gen_0001.jpg
  kitchen_gen_0050.jpg
  kitchen_0067.jpg
  kitchen_gen_0059.jpg
  kitchen_0028.jpg
  kitchen_0003.jpg
  kitchen_gen_0056.jpg
  kitchen_0042.jpg
  kitchen_gen_0024.jpg
  kitchen_0012.jpg
  kitchen_0022.jpg
  kitchen_gen_0075.jpg
  kitchen_0055.jpg
  kitchen_gen_0039.jpg
  kitchen_0010.jpg
  kitchen_gen_0042.jpg
  kitchen_0041.jpg
  kitchen_gen_0015.jpg
  kitchen_gen_0003.jpg
  kitchen_gen_0008.jpg
  kitchen_gen_0064.jpg
  kitchen_gen_0002.jpg
  kitchen_0044.jpg
  kitchen_gen_0054.jpg
  kitchen_gen_0011.jpg
  kitchen_gen_0007.jpg
  kitchen_0024.jpg
  kitchen_gen_0083.jpg
  kitchen_gen_0033.jpg
  kitchen_gen_0026.jpg
  kitchen_gen_0060.jpg
  kitchen_0049.jpg
  kitchen_0004.jpg
  kitchen_gen_0019.jpg
  kitchen_gen_0062.jpg
  kitchen_gen_0048.jpg
  kitchen_0019.jpg
  kitchen_gen_0044.jpg
  kitchen_gen_0004.jpg
  kitchen_gen_0061.jpg
  kitchen_gen_0013.jpg
  kitchen_gen_0036.jpg
  kitchen_gen_0034.jpg
  kitchen_0047.jpg
  kitc

In [ ]:
# ── PATCH: add skip-if-already-done logic (safe to run multiple times) ───────
import re, pathlib
p = pathlib.Path(f'{CODE_DIR}/scripts/render_batch_colab.py')
code = p.read_text()
if 'already_done' in code:
    print('Already patched')
else:
    old = '        print(f"[{ki}/{len(session_kitchens)}] {kitchen_path.name}  ({split})")\n\n        depth_cache'
    new = '        key = kitchen_path.name\n        already_done = len(manifest["kitchens"].get(key, {}).get("job_ids", []))\n        if already_done >= args.n:\n            print(f"[{ki}/{len(session_kitchens)}] {kitchen_path.name} [skip]")\n            total_skipped += 1\n            continue\n        print(f"[{ki}/{len(session_kitchens)}] {kitchen_path.name}  ({split})")\n\n        depth_cache'
    patched = code.replace(old, new).replace('    total_ok = 0\n    for ki', '    total_ok = 0\n    total_skipped = 0\n    for ki')
    p.write_text(patched)
    print('Patched!' if 'already_done' in patched else 'ERROR: patch failed - string not found')

Already patched


In [ ]:
# ── 6. RUN BATCH RENDERING ───────────────────────────────────────────────────
# This is the main cell. Runtime: ~1-2 hrs per session.
# Videos + annotations saved directly to Google Drive as they complete.
# Safe to re-run if interrupted — manifest tracks completed jobs.

!cd {CODE_DIR} && PYTHONPATH="generator/mmcv_stub:." python -u scripts/render_batch_colab.py \
    --image_dir      {IMAGE_DIR} \
    --depth_cache    {DEPTH_CACHE} \
    --output_dir     {OUTPUT_DIR} \
    --n              {VIDEOS_PER_KITCHEN} \
    --seed           {SPLIT_SEED} \
    --session_id     {SESSION_ID} \
    --total_sessions {TOTAL_SESSIONS}

Kitchen split summary:
  Total kitchens:   133
  Train:            107 kitchens × 20 = 2140 videos
  Val:              13 kitchens × 20 = 260 videos
  Held-out (test):  13 kitchens (not rendered in main run)

This session (2/5): 24 kitchens to render
Depth cache: enabled
Output: /content/drive/MyDrive/pest_project/renders

[1/24] kitchen_gen_0001.jpg [skip]
[2/24] kitchen_0003.jpg [skip]
[3/24] kitchen_0022.jpg [skip]
[4/24] kitchen_gen_0042.jpg [skip]
[5/24] kitchen_gen_0064.jpg [skip]
[6/24] kitchen_gen_0007.jpg [skip]
[7/24] kitchen_gen_0060.jpg [skip]
[8/24] kitchen_gen_0048.jpg [skip]
[9/24] kitchen_gen_0013.jpg [skip]
[10/24] kitchen_gen_0012.jpg [skip]
[11/24] kitchen_gen_0058.jpg [skip]
[12/24] kitchen_gen_0031.jpg [skip]
[13/24] kitchen_gen_0017.jpg [skip]
[14/24] kitchen_gen_0052.jpg [skip]
[15/24] kitchen_0031.jpg [skip]
[16/24] kitchen_gen_0076.jpg [skip]
[17/24] kitchen_0045.jpg [skip]
[18/24] kitchen_0069.jpg [skip]
[19/24] kitchen_0025.jpg [skip]
[20/24] kitchen_0026.jpg

In [ ]:
import json, os
from datetime import datetime, timedelta

manifest_path = f'{OUTPUT_DIR}/manifest.json'

if not os.path.exists(manifest_path):
    print('Manifest file not found. Rendering might not have started or OUTPUT_DIR is incorrect.')
else:
    m = json.load(open(manifest_path))

    kitchens = m.get('kitchens', {})
    n_per = m.get('n_per_kitchen', 20)

    print(f"Kitchens recorded in manifest: {len(kitchens)}")
    print(f"Total videos rendered so far:  {sum(len(v['job_ids']) for v in kitchens.values())}")

    # Calculate total videos expected to be rendered
    # From the prompt: 'Train: 107 kitchens × 20 = 2140 videos', 'Val: 13 kitchens × 20 = 260 videos'
    videos_total = (107 * n_per) + (13 * n_per)   # train + val kitchens × n_per videos each

    videos_done  = sum(len(v['job_ids']) for v in kitchens.values())
    videos_left  = max(0, videos_total - videos_done)

    sec_per_video = 20   # adjust if you know actual speed, default from notebook
    sessions_running = 3 # adjust based on how many Colab tabs you have open

    if videos_left > 0 and sessions_running > 0:
        eta_sec = (videos_left * sec_per_video) / sessions_running
        eta = timedelta(seconds=int(eta_sec))
        print(f"Videos done:    {videos_done} / {videos_total}")
        print(f"Videos left:    {videos_left}")
        print(f"ETA (~{sessions_running} sessions): {eta}")
    else:
        print(f"Videos done:    {videos_done} / {videos_total}")
        print("All videos rendered or no videos left to process for the configured sessions.")

Kitchens recorded in manifest: 120
Total videos rendered so far:  2400
Videos done:    2400 / 2400
All videos rendered or no videos left to process for the configured sessions.


In [ ]:
import json, os
from datetime import datetime, timedelta

manifest_path = f'{OUTPUT_DIR}/manifest.json'
m = json.load(open(manifest_path))

kitchens = m.get('kitchens', {})
n_per = m.get('n_per_kitchen', 20)

total_assigned = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}
total_done     = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}

# Infer which session rendered each kitchen from job count
# (we can't know for sure, but count by split)
by_split = {}
for k, v in kitchens.items():
    s = v['split']
    by_split[s] = by_split.get(s, 0) + len(v['job_ids'])

print(f"Kitchens recorded in manifest: {len(kitchens)}")
print(f"Total videos rendered so far:  {sum(len(v['job_ids']) for v in kitchens.values())}")
print(f"By split: {by_split}")
print()

# ETA estimate: ~20s per video on CPU
videos_done  = sum(len(v['job_ids']) for v in kitchens.values())
videos_total = 107 * 20 + 13 * 20   # train + val kitchens × 20 videos each
videos_left  = max(0, videos_total - videos_done)
sec_per_video = 20   # adjust if you know actual speed
sessions_running = 3

eta_sec = (videos_left * sec_per_video) / sessions_running
eta = timedelta(seconds=int(eta_sec))
print(f"Videos done:    {videos_done} / {videos_total}")
print(f"Videos left:    {videos_left}")
print(f"ETA (~{sessions_running} sessions): {eta}")

Kitchens recorded in manifest: 120
Total videos rendered so far:  2400
By split: {'train': 2140, 'val': 260}

Videos done:    2400 / 2400
Videos left:    0
ETA (~3 sessions): 0:00:00


In [ ]:
# ── 7. Check progress after rendering ────────────────────────────────────────
import json, os

manifest_path = f'{OUTPUT_DIR}/manifest.json'
if not os.path.exists(manifest_path):
    print('No manifest yet — rendering not started or output_dir wrong.')
else:
    m = json.load(open(manifest_path))
    kitchens_done = len(m.get('kitchens', {}))
    total_videos  = sum(len(v['job_ids']) for v in m['kitchens'].values())
    by_split = {}
    for info in m['kitchens'].values():
        s = info['split']
        by_split[s] = by_split.get(s, 0) + len(info['job_ids'])
    print(f'Kitchens rendered: {kitchens_done}')
    print(f'Total videos:      {total_videos}')
    print(f'By split:          {by_split}')

Kitchens rendered: 120
Total videos:      2400
By split:          {'train': 2140, 'val': 260}


In [ ]:
# ── MANIFEST CLEANUP — run ONCE after all sessions finish ────────────────────
import json, shutil
p = f'{OUTPUT_DIR}/manifest.json'
m = json.load(open(p))
n = m.get('n_per_kitchen', 20)
before = sum(len(v['job_ids']) for v in m['kitchens'].values())
for k in m['kitchens']:
    m['kitchens'][k]['job_ids'] = m['kitchens'][k]['job_ids'][:n]
after = sum(len(v['job_ids']) for v in m['kitchens'].values())
shutil.copy(p, p + '.bak')
open(p, 'w').write(json.dumps(m, indent=2))
print(f'Before: {before}  |  Removed: {before-after} duplicates  |  After: {after}')

Before: 2400  |  Removed: 0 duplicates  |  After: 2400


In [ ]:
!cd {CODE_DIR} && python scripts/build_dataset.py \
    --render_dir {OUTPUT_DIR} \
    --output_dir {DRIVE_ROOT}/pest_dataset \
    --every_n 2

Manifest: seed=42  n_per_kitchen=20  kitchens=120
  [WARN] Missing dirs for job f00b38d3 — skipping

Active splits: ['train', 'val']
  train: 2139 jobs
  val: 260 jobs

Processing train...
Traceback (most recent call last):
  File "/content/synthetic_video_gen/scripts/build_dataset.py", line 332, in <module>
    main()
  File "/content/synthetic_video_gen/scripts/build_dataset.py", line 299, in main
    process_split(split, split_jobs[split], output_dir, args.every_n, stats)
  File "/content/synthetic_video_gen/scripts/build_dataset.py", line 185, in process_split
    shutil.copy2(rec["frame_path"], img_dest)
  File "/usr/lib/python3.12/shutil.py", line 475, in copy2
    copyfile(src, dst, follow_symlinks=follow_symlinks)
  File "/usr/lib/python3.12/shutil.py", line 262, in copyfile
    with open(dst, 'wb') as fdst:
         ^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/pest_project/pest_dataset/images/train/frames_frame_0225.jpg'


## After ALL 5 sessions finish

Run this **once** in any session to build the final YOLO dataset:

```python
!cd {CODE_DIR} && python scripts/build_dataset.py \
    --render_dir {OUTPUT_DIR} \
    --output_dir {DRIVE_ROOT}/pest_dataset \
    --every_n 2
```

Then train:

```python
!pip install -q ultralytics
!yolo train \
    data={DRIVE_ROOT}/pest_dataset/data.yaml \
    model=yolov8m.pt \
    epochs=100 imgsz=640 batch=16 device=0 \
    project={DRIVE_ROOT}/pest_runs name=yolov8m_pest_v1 \
    augment=True mosaic=1.0 degrees=10 fliplr=0.5 \
    hsv_h=0.015 hsv_s=0.5 hsv_v=0.4 translate=0.1 scale=0.3
```